# 🧠 Create Vector Search Endpoint + Index (Python SDK)

This notebook programmatically sets up the infrastructure to support semantic search using **Cohere embeddings** and **Databricks Vector Search**.

### What This Does:
- ✅ Creates a **Vector Search endpoint** (if it doesn’t already exist)
- ✅ Registers a **Delta Sync index** using the chunked scouting report table
- ✅ Configures **Cohere `embed-english-v3`** as the embedding model
- ✅ Optionally waits for the index to be fully ready before continuing

Using the Python SDK ensures this process is reproducible and version-controlled — ideal for real production workflows or CI-driven pipelines.


In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Environment Variables
CATALOG = "alexander_booth"
SCHEMA = "cohere_demo"
TABLE = "fangraphs_mlb_scouting_reports"
TABLE_CHUNKS = "fangraphs_mlb_scouting_reports_chunked"

# Vector Store Environment Variables
INDEX_NAME = "fangraphs_mlb_scouting_reports_index"
EMBEDDING_MODEL = "cohere_embed_english_v3"  # your model serving endpoint
VECTOR_SEARCH_ENDPOINT = "vs_scouting_reports_demo"
PRIMARY_KEY = "primary_key"
SOURCE_COLUMN = "content_chunk"

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

In [0]:
# Create Vector Search Endpoint

# Check if endpoint exists
existing_endpoints = vsc.list_endpoints()
if VECTOR_SEARCH_ENDPOINT not in [ep["name"] for ep in existing_endpoints.get("endpoints", [])]:
  # Create endpoint
  vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT)
  print(f"✅ Endpoint '{VECTOR_SEARCH_ENDPOINT}' created.")
else:
  print(f"✅ Endpoint '{VECTOR_SEARCH_ENDPOINT}' already exists.")

✅ Endpoint 'vs_scouting_reports_demo' already exists.


In [0]:
# Create Vector Search Index

# Build the full index name
FULL_INDEX_NAME = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"
existing_indexes = vsc.list_indexes(name=VECTOR_SEARCH_ENDPOINT)

if FULL_INDEX_NAME not in [idx["name"] for idx in existing_indexes.get("vector_indexes", [])]:
  # Create Index
  vsc.create_delta_sync_index(
      index_name=FULL_INDEX_NAME,
      source_table_name=f"{CATALOG}.{SCHEMA}.{TABLE_CHUNKS}",
      pipeline_type="TRIGGERED",  # or "CONTINUOUS"
      primary_key=PRIMARY_KEY,
      embedding_source_column=SOURCE_COLUMN,
      embedding_model_endpoint_name=EMBEDDING_MODEL,
      endpoint_name=VECTOR_SEARCH_ENDPOINT
  )
  print(f"✅ Index '{FULL_INDEX_NAME}' created.")
else:
  print(f"✅ Index '{FULL_INDEX_NAME}' already exists.")

✅ Index 'alexander_booth.cohere_demo.fangraphs_mlb_scouting_reports_index' created.


In [0]:
# Retrieve the index object
index = vsc.get_index(endpoint_name=VECTOR_SEARCH_ENDPOINT, index_name=FULL_INDEX_NAME)

# Describe the index to get its status
status_info = index.describe()
print(f"Index status: {status_info['status']}")

Index status: {'detailed_state': 'PROVISIONING_ENDPOINT', 'message': 'Delta sync index creation is pending endpoint provisioning.', 'ready': False, 'index_url': 'e2-demo-field-eng.cloud.databricks.com/api/2.0/vector-search/indexes/alexander_booth.cohere_demo.fangraphs_mlb_scouting_reports_index'}


In [0]:
# Wait for the index to be ready
index.wait_until_ready()
print("✅ Index is ready for queries.")

✅ Index is ready for queries.


In [0]:
# Describe the index to get its status
status_info = index.describe()
print(f"Index status: {status_info['status']}")

Index status: {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://e2-demo-field-eng.cloud.databricks.com/explore/data/alexander_booth/cohere_demo/fangraphs_mlb_scouting_reports_index', 'indexed_row_count': 1119, 'triggered_update_status': {'last_processed_commit_version': 1, 'last_processed_commit_timestamp': '2025-05-21T17:36:47Z'}, 'ready': True, 'index_url': 'e2-demo-field-eng.cloud.databricks.com/api/2.0/vector-search/indexes/alexander_booth.cohere_demo.fangraphs_mlb_scouting_reports_index'}


---

✅ **Next Step: Run Semantic Search Queries**

With the index created and synced, you're ready to perform vector searches using:

- **Natural language phrases** via `vector_search(..., query_text => ...)`
- **Embeddings from existing reports** via `query_vector => ARRAY(...)`

You can run these queries directly in **SQL** or using the **Python SDK**.  
For example:
- Find players with similar scouting reports
- Search for traits like "elite bat speed and poor offspeed recognition"

👉 Continue to the next notebook to explore both use cases.